In [1]:


import os
import math

import torch
import torch.nn as nn
from tokenizers import Tokenizer                     # 分词工具
from torchtext.vocab import build_vocab_from_iterator    # 构建词典
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
from torch.nn.functional import pad, log_softmax   # pad用于文本对齐
from transformers import AutoTokenizer

import matplotlib.pyplot as plt
import numpy as np

# ========== 选择设备（优先GPU，无则用CPU） ==========
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"当前使用设备: {device}")  # 输出应显示cuda:0（GPU）或cpu

# 加载基础的分词器模型，使用的是基础的bert模型。`uncased`意思是不区分大小写
tokenizer = AutoTokenizer.from_pretrained("google-bert/bert-base-uncased")

# 分词封装
def en_tokenizer(line):
    """
    定义英文分词器
    :param line: 一句英文句子，例如"I'm learning Deep learning."
    :return: subword分词后的结果，例如：['i', "'", 'm', 'learning', 'deep', 'learning', '.']
    """
    # 使用bert进行分词，直接获取tokens
    return tokenizer.tokenize(line)

en_filepath = r"D:\homework\homework\Summer\files\train.en"
def yield_en_tokens():
    """
    每次yield一个分词后的英文句子，之所以yield方式是为了节省内存。
    如果先分好词再构造词典，那么将会有大量文本驻留内存，造成内存溢出。
    """
    with open(en_filepath, encoding='utf-8') as fd:
        for line in fd:
            yield en_tokenizer(line)

en_tok = yield_en_tokens()
for t in  en_tok:
    print(t)
    break

en_vocab_file = "D:/homework/homework/Summer/files/vocab_en.pt"

# 1. 提取目录路径
# en_vocab_dir = os.path.dirname(en_vocab_file)  # 结果为："D:/homework/homework/Summer/files"

# 2. 自动创建目录
# exist_ok=True 表示如果目录已存在，不会报错
# os.makedirs(en_vocab_dir, exist_ok=True)

# en_vocab = build_vocab_from_iterator(
#     en_tok,      # 可迭代的数据集分词列表
#     min_freq=2,  # 最小频率为2，即一个单词最少出现两次才会被收录到词典
#     specials=["<s>", "</s>", "<pad>", "<unk>"]) # 在词典的最开始加上这些特殊词。

# 设置词典的默认index，后面文本转index时，如果找不到，就会用该index填充
# en_vocab.set_default_index(en_vocab["<unk>"])
# torch.save(en_vocab, en_vocab_file) # 保存到文件

zh_filepath = r"D:\homework\homework\Summer\files\train.zh"
def zh_tokenizer(line):
    """
    定义中文分词器
    :param line: 中文句子，例如：机器学习
    :return: 分词结果，例如['机','器','学','习']
    """
    return list(line.strip().replace(" ", ""))


def yield_zh_tokens():
    with open(zh_filepath, encoding='utf-8') as fd:
        for line in fd:
            yield zh_tokenizer(line)

zh_tok = yield_zh_tokens()
for t in  zh_tok:
    print(t)
    break

zh_vocab_file = "D:/homework/homework/Summer/files/vocab_zh.pt"

# zh_vocab = build_vocab_from_iterator(
#     zh_tok,  
#     min_freq=1,
#     specials=["<s>", "</s>", "<pad>", "<unk>"])
# zh_vocab.set_default_index(zh_vocab["<unk>"])
# torch.save(zh_vocab, zh_vocab_file)


# 加载英文词典
en_vocab = torch.load(en_vocab_file)
print(f"✅ 成功加载英文词典，词典大小：{len(en_vocab)}")

# 加载中文词典
zh_vocab = torch.load(zh_vocab_file)
print(f"✅ 成功加载中文词典，词典大小：{len(zh_vocab)}")
# ----------------------------------------------------------------------------

print("中文词典大小:", len(zh_vocab))
print(dict((i, zh_vocab.lookup_token(i)) for i in range(10)))

class TranslationDataset(Dataset):

    def __init__(self):
        # 加载英文tokens
        self.en_tokens = self.load_tokens(en_filepath, en_tokenizer, en_vocab, 'en')
        # 加载中文tokens
        self.zh_tokens = self.load_tokens(zh_filepath, zh_tokenizer, zh_vocab, 'zh')

        self.row_count = len(self.zh_tokens)

    def __getitem__(self, index):
        return self.en_tokens[index], self.zh_tokens[index]

    def __len__(self):
        return self.row_count

    def load_tokens(self, file, tokenizer, vocab, lang):
        """
        加载tokens，即将文本句子们转换成index们。
        :param file: 文件路径，例如"./dataset/train.en"
        :param tokenizer: 分词器，例如en_tokenizer函数
        :param vocab: 词典, Vocab类对象。例如 en_vocab
        :param lang: 语言。用于构造缓存文件时进行区分。例如：’en‘
        :return: 返回构造好的tokens。例如：[[6, 8, 93, 12, ..], [62, 891, ...], ...]
        """
        # 从0开始构建，定义tokens_list用于存储结果
        tokens_list = []
        # 打开文件
        with open(file, encoding='utf-8') as fd:
            # 逐行读取
            for line in fd:
                # 进行分词
                tokens = tokenizer(line) 
                # 将文本分词结果通过词典转成index
                tokens = vocab(tokens)    # 使用词典对象对分词文本进行数值化。
                # append到结果中
                tokens_list.append(tokens)

        return tokens_list

ds = TranslationDataset()

print(ds[0])




当前使用设备: cuda:0


D:\anaconder\envs\trans_env\lib\site-packages\huggingface_hub\file_download.py:945: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


['a', 'pair', 'of', 'red', '-', 'crowned', 'cranes', 'have', 'stake', '##d', 'out', 'their', 'nesting', 'territory']
['一', '对', '丹', '顶', '鹤', '正', '监', '视', '着', '它', '们', '的', '筑', '巢', '领', '地']
✅ 成功加载英文词典，词典大小：27584
✅ 成功加载中文词典，词典大小：8280
中文词典大小: 8280
{0: '<s>', 1: '</s>', 2: '<pad>', 3: '<unk>', 4: '。', 5: '的', 6: '，', 7: '我', 8: '你', 9: '是'}
([11, 2731, 12, 554, 19, 17230, 18104, 27, 3081, 203, 57, 102, 18856, 3653], [12, 40, 1173, 1084, 3169, 164, 693, 397, 84, 100, 14, 5, 1218, 2398, 535, 67])


In [16]:
max_length = 72
def collate_fn(batch):
    """
    将dataset的数据进一步处理，并组成一个batch。
    :param batch: 一个batch的数据，例如：
                  [([6, 8, 93, 12, ..], [62, 891, ...]),
                  ....
                  ...]
    :return: 填充后的且等长的数据，包括src, tgt, tgt_y, n_tokens
             其中src为原句子，即要被翻译的句子
             tgt为目标句子：翻译后的句子，但不包含最后一个token
             tgt_y为label：翻译后的句子，但不包含第一个token，即<bos>
             n_tokens：tgt_y中的token数，<pad>不计算在内。
    """
    # 定义'<bos>'的index，在词典中为0，所以这里也是0
    bs_id = torch.tensor([0])
    # 定义'<eos>'的index
    eos_id = torch.tensor([1])
    # 定义<pad>的index
    pad_id = 2
    # 用于存储处理后的src=en和tgt=zh
    src_list, tgt_list = [], []

    # 循环遍历句子对儿
    for (_src, _tgt) in batch:
        """
        _src: 英语句子，例如：`I love you`对应的index
        _tgt: 中文句子，例如：`我 爱 你`对应的index
        """
        # 将<bos>，句子index和<eos>拼到一块
        processed_src = torch.cat([bs_id, torch.tensor(_src, dtype=torch.int64), eos_id], dim=0) # 按照行链接
        processed_tgt = torch.cat([bs_id, torch.tensor(_tgt, dtype=torch.int64), eos_id], dim=0)

        """
        将长度不足的句子进行填充到max_padding的长度的，然后增添到list中

        pad：假设processed_src为[0, 1136, 2468, 1349, 1]
             第二个参数为: (0, 72-5)
             第三个参数为：2
        则pad的意思表示，给processed_src左边填充0个2，右边填充67个2。
        最终结果为：[0, 1136, 2468, 1349, 1, 2, 2, 2, ..., 2]
        """
        processed_src = pad(processed_src, (0, max_length - len(processed_src)), value=pad_id)
        src_list.append(processed_src)
        processed_tgt = pad(processed_tgt, (0, max_length - len(processed_tgt)), value=pad_id)
        tgt_list.append(processed_tgt)

    # 将多个src句子堆叠到一起
    src = torch.stack(src_list)
    tgt = torch.stack(tgt_list)

    # tgt_y是目标句子去掉第一个token，即去掉<bos>
    tgt_y = tgt[:, 1:]
    # tgt是目标句子去掉最后一个token
    tgt = tgt[:, :-1]

    # 计算本次batch要预测的token数
    n_tokens = (tgt_y != 2).sum()

    # 返回batch后的结果
    return src, tgt, tgt_y, n_tokens

train_loader = DataLoader(ds, batch_size=16, shuffle=True, collate_fn=collate_fn) #一个批次64个样本
for src, tgt, tgt_y, n_tokens in train_loader:
    print(src.shape, tgt.shape, tgt_y.shape, n_tokens)
    break



class PositionalEncoding(nn.Module):
    def __init__(self, d_model, dropout, max_len=5000):
        super(PositionalEncoding, self).__init__()
        self.dropout = nn.Dropout(p=dropout)
        # 初始化Shape为(max_len, d_model)的positional encoding
        pe = torch.zeros(max_len, d_model)
        # 初始化一个tensor [[0, 1, 2, 3, ...]]
        position = torch.arange(0, max_len).unsqueeze(1)
        # 这里就是sin和cos括号中的内容，通过e和ln进行了变换
        div_term = torch.exp(
            torch.arange(0, d_model, 2) * -(math.log(10000.0) / d_model)
        )
        # 计算PE(pos, 2i)
        pe[:, 0::2] = torch.sin(position * div_term)
        # 计算PE(pos, 2i+1)
        pe[:, 1::2] = torch.cos(position * div_term)
        # 为了方便计算，在最外面在unsqueeze出一个batch
        pe = pe.unsqueeze(0)
        # 如果一个参数不参与梯度下降，但又希望保存model的时候将其保存下来
        # 这个时候就可以用register_buffer
        self.register_buffer("pe", pe)

    def forward(self, x):
        """
        x 为embedding后的inputs，例如(1,7, 128)，batch size为1,7个单词，单词维度为128
        """
        # 将x和positional encoding相加。
        x = x + self.pe[:, : x.size(1)].requires_grad_(False)
        return self.dropout(x)




# --------------------------
# 模型定义
# --------------------------
class TranslationModel(nn.Module):
    def __init__(self, d_model, src_vocab, tgt_vocab, dropout=0.1):
        super(TranslationModel, self).__init__()
        self.src_embedding = nn.Embedding(len(src_vocab), d_model, padding_idx=2)
        self.tgt_embedding = nn.Embedding(len(tgt_vocab), d_model, padding_idx=2)
        self.positional_encoding = PositionalEncoding(d_model, dropout, max_len=max_length)
        self.transformer = nn.Transformer(
            d_model, 
            dropout=dropout, 
            batch_first=True, 
            nhead=8, 
            num_encoder_layers=2, 
            num_decoder_layers=2, 
            dim_feedforward=128)
        self.predictor = nn.Linear(d_model, len(tgt_vocab))

    def forward(self, src, tgt):
        """原有训练用forward：未改动"""
        tgt_mask = nn.Transformer.generate_square_subsequent_mask(tgt.size()[-1]).to(src.device)
        src_key_padding_mask = TranslationModel.get_key_padding_mask(src).float().to(src.device)
        tgt_key_padding_mask = TranslationModel.get_key_padding_mask(tgt).float().to(src.device)

        src = self.src_embedding(src)
        tgt = self.tgt_embedding(tgt)
        src = self.positional_encoding(src)
        tgt = self.positional_encoding(tgt)
        
        out = self.transformer(
            src, tgt,
            tgt_mask=tgt_mask,
            src_key_padding_mask=src_key_padding_mask,
            tgt_key_padding_mask=tgt_key_padding_mask
        )
        return out

    @staticmethod
    def get_key_padding_mask(tokens):
        return tokens == 2

    # 新增：自回归生成函数（用于多段翻译推理）
    def generate(self, src, max_gen_len=72):
        """
        单段文本的自回归生成
        :param src: 单段输入张量，shape=(1, seq_len)
        :return: 生成的目标语言token列表（不含<bos>）
        """
        self.eval()  # 推理模式
        with torch.no_grad():
            # 1. 编码器处理src
            src_key_padding_mask = self.get_key_padding_mask(src).float().to(src.device)
            src_emb = self.src_embedding(src)
            src_pe = self.positional_encoding(src_emb)
            memory = self.transformer.encoder(src_pe, src_key_padding_mask=src_key_padding_mask)
            
            # 2. 初始化解码器输入（仅含<bos>）
            tgt_pred = torch.tensor([[0]], dtype=torch.int64).to(src.device)  # <bos>的索引为0
            
            # 3. 自回归生成
            for _ in range(max_gen_len):
                # 生成解码器掩码
                tgt_mask = nn.Transformer.generate_square_subsequent_mask(tgt_pred.size(1)).to(src.device)
                tgt_key_padding_mask = self.get_key_padding_mask(tgt_pred).float().to(src.device)
                
                # 解码器前向计算
                tgt_emb = self.tgt_embedding(tgt_pred)
                tgt_pe = self.positional_encoding(tgt_emb)
                out = self.transformer.decoder(
                    tgt_pe, memory,
                    tgt_mask=tgt_mask,
                    tgt_key_padding_mask=tgt_key_padding_mask,
                    memory_key_padding_mask=src_key_padding_mask
                )
                
                # 预测下一个token
                logits = self.predictor(out[:, -1, :])
                next_token = sample_top_k(logits, k=5) 
                tgt_pred = torch.cat([tgt_pred, next_token], dim=1)
                
                # 遇到<eos>停止
                if next_token.item() == 1:  # <eos>的索引为1
                    break
            
            # 4. 处理输出：去掉<bos>和<eos>
            tgt_pred = tgt_pred.squeeze(0).tolist()[1:]  # 去掉batch维度和<bos>
            if 1 in tgt_pred:  # 去掉<eos>及之后的内容
                tgt_pred = tgt_pred[:tgt_pred.index(1)]
            return tgt_pred


# 初始化模型
model = TranslationModel(200, en_vocab, zh_vocab)
# 模型迁移到GPU
model = model.to(device)  # 关键：将模型参数移到GPU

for src, tgt, tgt_y, n_tokens in train_loader:
    # 数据迁移到GPU
    src = src.to(device)
    tgt = tgt.to(device)
    y = model(src, tgt)
    print(y.shape)
    break

optimizer = torch.optim.Adam(model.parameters(), lr=5e-5)

class TranslationLoss(nn.Module):

    def __init__(self):
        super(TranslationLoss, self).__init__()
        self.criterion = nn.KLDivLoss(reduction="sum")
        self.padding_idx = 2  # <pad>的索引

    def forward(self, x, target):
        # 1. 对模型输出做log_softmax（保持在原设备）
        x = log_softmax(x, dim=-1)
        
        # 2. 关键修复：创建true_dist时强制指定与x相同的设备（GPU）
        # 原错误：true_dist默认在CPU，这里显式用x的设备
        true_dist = torch.zeros(x.size(), device=x.device)  
        
        # 3. 用target的索引填充true_dist（此时两者设备一致）
        true_dist.scatter_(1, target.data.unsqueeze(1), 1.0)
        
        # 4. 处理<pad>位置（mask会自动继承true_dist的设备）
        mask = torch.nonzero(target.data == self.padding_idx)
        if mask.dim() > 0:
            true_dist.index_fill_(0, mask.squeeze(), 0.0)
        
        # 5. 计算损失（所有张量设备已同步）
        return self.criterion(x, true_dist.clone().detach())


# --------------------------
# 多段文本处理工具
# --------------------------
def multi_seg_tokenize(src_multi, src_vocab, max_length):
    """
    将多段文本转换为单段样本列表
    :param src_multi: 多段输入文本，如["I love you.", "He is happy."]
    :param src_vocab: 源语言词典
    :param max_length: 最大序列长度
    :return: 单段样本列表，可直接送入DataLoader
    """
    single_samples = []
    for seg in src_multi:
        # 1. 分词
        tokens = seg.strip().split()  # 示例：空格分词
        # 2. 转为索引（未知词用<unk>，假设<unk>索引为3）
        seg_idx = [src_vocab.get(token, 3) for token in tokens]
        # 3. 截断超长文本（预留<bos>和<eos>的位置）
        if len(seg_idx) > max_length - 2:
            seg_idx = seg_idx[:max_length - 2]
        # 4. 组成样本（tgt用占位符，不影响推理）
        single_samples.append( (seg_idx, [0]) )  # (src索引, 占位tgt)
    return single_samples


# --------------------------
# 多段翻译推理函数
# --------------------------
def translate_multi_seg(src_multi, model, src_vocab, tgt_vocab, max_length, device):
    """
    多段文本翻译主函数
    :param src_multi: 多段输入文本列表
    :return: 多段翻译结果列表
    """
    # 1. 多段转单段样本
    single_samples = multi_seg_tokenize(src_multi, src_vocab, max_length)
    # 2. 用原有collate_fn批量处理
    loader = DataLoader(
        single_samples,
        batch_size=len(single_samples),  # 一次处理所有段
        collate_fn=collate_fn
    )
    # 3. 模型推理
    model.eval()
    translations = []
    with torch.no_grad():
        for src, _, _, _ in loader:  # 只需要src
            src = src.to(device)
            # 逐段生成翻译
            for i in range(src.shape[0]):
                single_src = src[i].unsqueeze(0)  # 单段输入，shape=(1, seq_len)
                pred_idx = model.generate(single_src, max_gen_len=max_length)
                # 索引转文本
                pred_text = "".join([tgt_vocab.idx_to_token[idx] for idx in pred_idx 
                                    if idx != 2])  # 过滤<pad>
                translations.append(pred_text)
    return translations


criteria = TranslationLoss()

# 初始化训练参数
epochs = 18  
start_epoch = 0  
resume_training = True  # 是否续训
checkpoint_path = "D:/homework/homework/Summer/files/model_best.pt"
best_loss = float('inf')
best_model_path = "D:/homework/homework/Summer/files/model_best.pt"
save_after_step = 100  
step = 0 

# 初始化损失记录和最优模型变量
train_step_losses = []  # 记录「每一步（每个batch）」的损失，用于绘制细粒度曲线
train_epoch_avg_losses = []  # 记录「每一轮」的平均损失，用于判断最优模型


if resume_training:
    try:
        checkpoint = torch.load(checkpoint_path, map_location=device)
        model.load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        start_epoch = checkpoint['epoch'] 
        best_loss = checkpoint['best_loss']
        
        # 加载损失记录
        train_step_losses = torch.load('step_losses.pt', map_location=device) if os.path.exists('step_losses.pt') else []
        train_epoch_avg_losses = torch.load('epoch_losses.pt', map_location=device) if os.path.exists('epoch_losses.pt') else []
        
        # 检查是否已完成所有轮次
        if start_epoch+1 >= epochs:
            print(f"已完成所有{epochs}轮训练，无需继续")
            # 可在这里直接退出或处理后续逻辑
        else:
            print(f"已加载检查点，从轮次 {start_epoch + 1}/{epochs} 继续训练") 
            model.train()
    except:
        print("未找到检查点，将从头开始训练")


for epoch in range(start_epoch + 1, epochs):  
    epoch_total_loss = 0.0
    epoch_batch_count = 0
    
    for index, data in enumerate(train_loader):
        # 数据处理与模型训练（保持不变）
        src, tgt, tgt_y, n_tokens = data
        src = src.to(device)
        tgt = tgt.to(device)
        tgt_y = tgt_y.to(device)
        
        optimizer.zero_grad()
        out = model(src, tgt)
        out = model.predictor(out)
        loss = criteria(out.contiguous().view(-1, out.size(-1)), tgt_y.contiguous().view(-1)) / n_tokens
        
        # 损失记录与反向传播
        train_step_losses.append(loss.item())
        epoch_total_loss += loss.item()
        epoch_batch_count += 1
        loss.backward()
        optimizer.step()
        
        # 中间日志
        step += 1
        if step % save_after_step == 0:
            print(f"步数：{step:04d}，当前步损失：{loss.detach().item():.4f}")        
    
    # 轮次结束处理
    epoch_avg_loss = epoch_total_loss / epoch_batch_count
    train_epoch_avg_losses.append(epoch_avg_loss)
    print(f"="*50)
    print(f"轮次：{epoch + 1}/{epochs}，本轮平均损失：{epoch_avg_loss:.4f}")  # 显示人类习惯的轮次
    print(f"当前最优损失：{best_loss:.4f}")

    # 保存损失记录
    torch.save(train_step_losses, 'step_losses.pt')
    torch.save(train_epoch_avg_losses, 'epoch_losses.pt')

    # 保存最优模型
    if epoch_avg_loss < best_loss:
        best_loss = epoch_avg_loss
        torch.save({
            "epoch": epoch,  
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "best_loss": best_loss
        }, best_model_path)
        print(f"✅ 轮次{epoch + 1}：更新最优模型，保存至 {best_model_path}")
    print(f"="*50 + "\n")


print(f"\n训练结束！")
print(f"2. 最优模型已保存至：{best_model_path}")
print(f"3. 最优损失：{best_loss:.4f}（对应轮次：{train_epoch_avg_losses.index(best_loss)+1}）")

torch.Size([16, 72]) torch.Size([16, 71]) torch.Size([16, 71]) tensor(378)
torch.Size([16, 71, 200])
已加载检查点，从轮次 17/18 继续训练
步数：0100，当前步损失：2.1421
步数：0200，当前步损失：1.9251
步数：0300，当前步损失：2.1935
步数：0400，当前步损失：1.7668
步数：0500，当前步损失：2.0931
步数：0600，当前步损失：2.0175
步数：0700，当前步损失：2.4548
步数：0800，当前步损失：2.2547
步数：0900，当前步损失：2.6153
步数：1000，当前步损失：2.1629
步数：1100，当前步损失：1.8141
步数：1200，当前步损失：2.3250
步数：1300，当前步损失：2.1347
步数：1400，当前步损失：1.4806
步数：1500，当前步损失：2.7209
步数：1600，当前步损失：2.3976
步数：1700，当前步损失：2.3796
步数：1800，当前步损失：1.8659
步数：1900，当前步损失：2.2533
步数：2000，当前步损失：2.3970
步数：2100，当前步损失：2.6718
步数：2200，当前步损失：2.0981
步数：2300，当前步损失：2.0181
步数：2400，当前步损失：1.6826
步数：2500，当前步损失：2.2705
步数：2600，当前步损失：2.2245
步数：2700，当前步损失：2.2147
步数：2800，当前步损失：2.4586
步数：2900，当前步损失：2.0252
步数：3000，当前步损失：2.3214
步数：3100，当前步损失：2.2762
步数：3200，当前步损失：2.6733
步数：3300，当前步损失：2.6002
步数：3400，当前步损失：2.1655
步数：3500，当前步损失：2.3549
步数：3600，当前步损失：2.2972
步数：3700，当前步损失：2.0147
步数：3800，当前步损失：2.2525
步数：3900，当前步损失：2.0903
步数：4000，当前步损失：2.0578
步数：4100，当前步损失：1.9974
步数：4200，当前步损失：2.

In [56]:
# 1. 加载训练好的最优模型
print("\n" + "="*60)
print("开始加载最优模型并准备翻译测试...")
# 重新初始化与训练一致的模型结构
infer_model = TranslationModel(d_model=200, src_vocab=en_vocab, tgt_vocab=zh_vocab)
# 加载最优模型参数
checkpoint = torch.load(best_model_path, map_location=device)
infer_model.load_state_dict(checkpoint['model_state_dict'])
# 切换为评估模式
infer_model = infer_model.eval()
# 移到指定设备
infer_model = infer_model.to(device)
print(f"✅ 最优模型加载完成，当前模式：eval，设备：{device}")

def sample_top_k(logits, k=5):
    topk_probs, topk_indices = torch.topk(torch.softmax(logits, dim=-1), k)
    idx = torch.multinomial(topk_probs, 1)  # 按概率抽样
    return topk_indices.gather(-1, idx)


# 2. 定义翻译函数
def translate(src: str):
    """
    :param src: 英文句子，例如 "I like machine learning."
    :return: 翻译后的句子，例如：”我喜欢机器学习“
    """
    # 英文句子分词 → 转词典index → 增加<BOS>(<s>, 0)和<EOS>(</s>, 1)
    src_tok = en_tokenizer(src)
    src_indices = [0] + en_vocab(src_tok) + [1]  # 拼接特殊符号
    # 转为tensor并增加batch维度（模型输入要求batch_first=True）
    src_tensor = torch.tensor(src_indices).unsqueeze(0).to(device)
    
    # 初始化目标语言输入：仅包含<BOS>（起始符号）
    tgt_tensor = torch.tensor([[0]]).to(device)  # shape: (1, 1)
    
    # 逐词预测：直到出现<EOS>或达到最大长度
    max_gen_len = min(max_length, len(src_indices) + 4)  # 限制生成长度，避免无限循环
    for _ in range(max_gen_len):
        # 模型前向传播（评估模式下无需计算梯度）
        with torch.no_grad():
            out = infer_model(src_tensor, tgt_tensor)
            # 取最后一个token的预测结果（因为是自回归生成）
            pred_logits = infer_model.predictor(out[:, -1])  # shape: (1, 中文词典大小)
            # 选择概率最大的token索引
            pred_token_idx = sample_top_k(pred_logits, k=5).squeeze(1)

        
        # 将预测的token拼接到目标序列中
        tgt_tensor = torch.concat([tgt_tensor, pred_token_idx.unsqueeze(0)], dim=1)
        
        # 若预测到<EOS>（索引1），停止生成
        if pred_token_idx.item() == 1:
            break
    
    # 将预测的token索引转为中文文本（清理特殊符号）
    tgt_tokens = zh_vocab.lookup_tokens(tgt_tensor.squeeze().tolist())
    # 移除<BOS>、<EOS>、<PAD>等特殊符号
    translated_text = ''.join([tok for tok in tgt_tokens 
                               if tok not in ["<s>", "</s>", "<pad>"]])
    return translated_text


# 3. 测试翻译功能
print("\n开始翻译测试：")
test_sentence = " a father and a son sitting in the cafe"
translated_result = translate(test_sentence)
print(f"英文原文：{test_sentence}")
print(f"中文译文：{translated_result}")
print("="*60)


开始加载最优模型并准备翻译测试...
✅ 最优模型加载完成，当前模式：eval，设备：cuda:0

开始翻译测试：
英文原文： a father and a son sitting in the cafe
中文译文：一名父亲和一个儿子在咖啡馆里坐
